# CU HXR BMAD Model Walkthrough

> This notebook demonstrates practical patterns for using the CU HXR model: initialization, variable I/O, matrix inspection, image diagnostics, parameter scans, and optics plotting.

In [ ]:
from virtual_accelerator.models.cu_hxr import get_cu_hxr_bmad_model
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 5)

In [ ]:
# Build the CU HXR model and select tracking mode
model = get_cu_hxr_bmad_model()
model.set({"track_type": 1})

# Keep frequently used PV names in one place
OTR_IMAGE_PV = "OTRS:IN20:711:Image:ArrayData"
SCAN_QUAD_PV = "QUAD:IN20:631:BCTRL"

def get_otr_image(active_model):
    """Return the 2D OTR image array for the configured screen PV."""
    return active_model.get([OTR_IMAGE_PV])[OTR_IMAGE_PV]

## 1) Inspect and Query Supported Variables

Start by checking how many variables are exposed, then query a compact subset of useful channels.

In [ ]:
variable_names = list(model.supported_variables.keys())
print(f"Supported variables: {len(variable_names)}")
print("Example variable names:")
print(variable_names[:10])

In [ ]:
selected_pvs = [
    SCAN_QUAD_PV,
    "OTR4_beam",
    "track_type",
]
model.get(selected_pvs)

In [ ]:
# Inspect one scalar-like channel
model.get(["OTR4_beam"])

## 2) Access Element Transport Matrices

You can retrieve element names and flattened 6x6 transport matrices, then reshape and inspect a specific element.

In [ ]:
matrix_info = model.get(["mat6", "name"])
names = matrix_info["name"]
mat6 = matrix_info["mat6"].reshape(-1, 6, 6)
target_name = "QE04#1"

target_index = list(names).index(target_name)
print(f"Found {target_name} at index {target_index}")
mat6[target_index]

## 3) Visualize Baseline OTR Image

Capture a baseline screen image and inspect quick image statistics before scanning parameters.

In [ ]:
baseline_image = get_otr_image(model)
print(
    f"Image stats -> min: {baseline_image.min():.3g}, "
    f"max: {baseline_image.max():.3g}, "
    f"mean: {baseline_image.mean():.3g}"
)

fig, ax = plt.subplots()
ax.imshow(baseline_image)
ax.set_title("Baseline OTR Image")
ax.set_xlabel("x [pixel]")
ax.set_ylabel("y [pixel]")

## 4) Scan an Upstream Quadrupole and Compare Images

This scan changes a quadrupole setpoint by scale factors, records resulting OTR images, and restores the original setpoint at the end.

In [ ]:
original_quad = model.get([SCAN_QUAD_PV])[SCAN_QUAD_PV]
scan_scales = [0.7, 0.85, 1.0, 1.15, 1.3]
scan_results = []

for scale in scan_scales:
    test_value = float(original_quad * scale)
    model.set({SCAN_QUAD_PV: test_value})
    image = get_otr_image(model)
    scan_results.append((scale, test_value, image))

# Restore nominal setting
model.set({SCAN_QUAD_PV: original_quad})
print(f"Restored {SCAN_QUAD_PV} = {original_quad:.6g}")

In [ ]:
fig, axes = plt.subplots(1, len(scan_results), figsize=(3 * len(scan_results), 3), constrained_layout=True)

for ax, (scale, value, image) in zip(axes, scan_results):
    ax.imshow(image)
    ax.set_title(f"scale={scale:.2f}\n{value:.3f}")
    ax.set_xlim(400, 600)
    ax.set_ylim(600, 800)

fig.suptitle("OTR Response to QUAD:IN20:631:BCTRL Scan", y=1.05)

## 5) Direct TAO Command Path

You can also modify model state with direct TAO commands, then sync observable state via `update_state()`.

In [ ]:
model.tao.cmd("set ele QM01 B1_GRADIENT = 3.5")
model.update_state()

tao_image = get_otr_image(model)
fig, ax = plt.subplots()
ax.imshow(tao_image)
ax.set_title("OTR Image After TAO Command")
ax.set_xticks([])
ax.set_yticks([])

## 6) Plot Twiss Functions Along the Beamline

Finally, fetch horizontal/vertical beta functions and plot them for a quick optics sanity check.

In [ ]:
twiss_info = model.get(["a.beta", "b.beta"])

fig, ax = plt.subplots()
ax.plot(twiss_info["a.beta"], label="beta_x")
ax.plot(twiss_info["b.beta"], label="beta_y")
ax.set_title("Twiss Beta Functions")
ax.set_xlabel("Element index")
ax.set_ylabel("Beta [m]")
ax.grid(True, alpha=0.3)
ax.legend()